# Chapter 2 (hadronic) — Notebook 4: Cut optimisation

**Goals**

- Optimise jet- and b-tag-count requirements for hadronic-side significance (weighted).

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()
samples = io.build_samples()
procs = ['ttbar', 'single_top', 'diboson']
evs = {p: io.load_process(p, samples, fraction=0.1) for p in procs}

import pandas as pd
rows = []
for nj in (4, 5, 6):
    for nb in (1, 2):
        cuts = selection.SemilepCuts(n_jets_min=nj, n_bjets_min=nb)
        y = {p: float(ak.sum(weights.weight_for(p, ev)[selection.semilep_preselection(ev, cuts)]))
             for p, ev in evs.items()}
        s, b = y['ttbar'], y['single_top'] + y['diboson']
        rows.append((nj, nb, plotting.significance([s], [b])))
pd.DataFrame(rows, columns=['n_jets_min', 'n_bjets_min', 'S/sqrt(S+B)'])

## ✏️ Your turn 4.1

▶️ Change the W-window width and re-run.

On top of the ≥4-jet, ≥2-b-tag selection, this also requires the two leading light jets to have
$|m_{jj} - m_W| <$ `MW_WINDOW` GeV (consistent with a hadronic W). It prints the weighted
significance $S/\sqrt{S+B}$ with and without the window. Does the W-window improve it?

> **Challenge (optional):** try several `MW_WINDOW` values and find the one that maximises significance.

In [ ]:
MW_WINDOW = 20.0    # ✏️ try 10, 20, 40 GeV

cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)

def yields(apply_window):
    y = {}
    for p, ev in evs.items():
        ev_sel = ev[selection.semilep_preselection(ev, cuts)]
        w = weights.weight_for(p, ev_sel)
        if apply_window:
            jets = kinematics.jet_vectors(ev_sel)
            light = jets[ev_sel.jet_btag_quantile < cuts.btag_quantile_min]
            keep = ak.num(light) >= 2
            m_jj = (light[keep][:, 0] + light[keep][:, 1]).mass
            in_w = abs(m_jj - M_W) < MW_WINDOW
            y[p] = float(ak.sum(w[keep][in_w]))
        else:
            y[p] = float(ak.sum(w))
    return y

for label, apply_w in [('no W-window', False), (f'|m_jj - m_W| < {MW_WINDOW:g}', True)]:
    y = yields(apply_w)
    s, b = y['ttbar'], y['single_top'] + y['diboson']
    print(f'{label:22s}  S/sqrt(S+B) = {plotting.significance([s], [b]):.2f}')